In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "/Users/xinc/GitHub/note")
sys.path.insert(0, "/Users/xinc/GitHub/Quant")
# Local project path is optional; notebook directory is already on sys.path.
# sys.path.insert(0, "D:/Github/Quant/projects/CTA/學弟策略")

from pathlib import Path
import gc
import time
from datetime import datetime
import pandas as pd
from tqdm import tqdm

from module.get_info_FinMind import FinMindClient
from utils import (
    chunk_list,
    compact_daily_price_parquet,
    get_valid_stock_ids,
    get_missing_kbar_ids,
    infer_stock_ids_from_kbar_dir,
    load_no_price_pairs,
    load_or_build_above_ma_signal,
    load_or_refresh_daily_prices,
    read_kbar_ids,
    read_kbar_with_supplement,
    build_kbar_coverage_report,
    summarize_kbar_coverage,
    signal_row_to_stock_ids,
    update_missing_kbar_ids,
    update_missing_full_kbar_dates,
)

client = FinMindClient()
from cloud_data import (
    DATA_ROOT,
    TRADING_ROOT,
    TW_STOCK_DAILY_PRICE,
    TW_STOCK_KBAR_1MIN,
    TW_STOCK_KBAR_ABOVE_MA60,
    TW_FUTURES_TX,
)

2026-07-14 20:58:46.939 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


## get data

### config

In [3]:
start_date = "2022-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")

ma_window = 60
retry_wait = 60
max_retries = 1
rate_limit_wait = 3660
rate_limit_max_retries = 2
api_batch_size = 30
rebuild_signal = True

# Fetch extra daily history so the first target date can have a valid MA60.
daily_start_date = (
    pd.Timestamp(start_date) - pd.Timedelta(days=ma_window * 3)
).strftime("%Y-%m-%d")

data_dir = TRADING_ROOT
stock_universe_output = DATA_ROOT / "trading" / "market_reference" / "note_data" / "stock_universe.parquet"
daily_output = TW_STOCK_DAILY_PRICE
full_kbar_dir = TW_STOCK_KBAR_1MIN
signal_output = full_kbar_dir / "above_ma60_signal.parquet"
kbar_output_dir = TW_STOCK_KBAR_ABOVE_MA60
missing_kbar_output = kbar_output_dir / "missing_kbar_ids.json"
missing_full_kbar_output = kbar_output_dir / "missing_full_kbar_dates.json"
kbar_output_dir.mkdir(parents=True, exist_ok=True)

### get daily data

In [6]:
# 1. Load or refresh raw daily prices for valid stocks only.
stock_ids = get_valid_stock_ids(client, output_file=stock_universe_output)
print(f"Valid stock count: {len(stock_ids):,}")

daily = load_or_refresh_daily_prices(
    client=client,
    output_file=daily_output,
    start_date=daily_start_date,
    end_date=end_date,
    stock_ids=stock_ids,
    retry_wait=retry_wait,
)

daily["date"] = pd.to_datetime(daily["date"])
print(f"Daily rows: {len(daily):,}")
print(f"Daily dates: {daily['date'].nunique():,}")
print(f"Daily stocks: {daily['stock_id'].astype(str).nunique():,}")
print(f"Daily range: {daily['date'].min().date()} ~ {daily['date'].max().date()}")

Valid stock count: 2,138


Download daily price: 100%|██████████| 30/30 [04:14<00:00,  8.48s/day]


Daily rows: 2,329,905
Daily dates: 1,220
Daily stocks: 2,052
Daily range: 2021-07-05 ~ 2026-07-13


In [17]:
# Optional: compact the old huge daily_stock_price.parquet without calling API.
old_daily_output = data_dir / "daily_stock_price.parquet"

if not old_daily_output.exists():
    raise FileNotFoundError(old_daily_output)

if stock_universe_output.exists():
    stock_ids_for_compact = pd.read_parquet(stock_universe_output)["stock_id"].astype(str)
    stock_ids_for_compact = stock_ids_for_compact[stock_ids_for_compact.str.fullmatch(r"[1-9]\d{3}")].tolist()
else:
    stock_ids_for_compact = infer_stock_ids_from_kbar_dir(full_kbar_dir)
    if not stock_ids_for_compact:
        raise FileNotFoundError(
            "No stock_universe.parquet and no kbar/1min parquet files to infer stock ids from."
        )
    pd.DataFrame({"stock_id": stock_ids_for_compact}).to_parquet(
        stock_universe_output,
        index=False,
    )

print(f"Compacting {len(stock_ids_for_compact):,} stock ids")

daily = compact_daily_price_parquet(
    source_file=old_daily_output,
    output_file=daily_output,
    stock_ids=stock_ids_for_compact,
)

print(f"Daily rows: {len(daily):,}")
print(f"Daily dates: {daily['date'].nunique():,}")
print(f"Daily stocks: {daily['stock_id'].astype(str).nunique():,}")
print(f"Daily range: {daily['date'].min().date()} ~ {daily['date'].max().date()}")

Compacting 2,138 stock ids


Compact daily price: 100%|██████████| 5/5 [00:00<00:00, 10.62batch/s]


Daily rows: 2,272,554
Daily dates: 1,191
Daily stocks: 2,022
Daily range: 2021-07-05 ~ 2026-05-29


### build 60ma signal

In [7]:
# 2. Build/load date x stock_id signal matrix.
# True means this stock passed yesterday's close > MA60 condition for today's kbar.
if "daily" not in globals():
    daily = pd.read_parquet(
        daily_output,
        columns=["date", "stock_id", "close"],
    )

signal = load_or_build_above_ma_signal(
    daily=daily,
    output_file=signal_output,
    start_date=start_date,
    end_date=end_date,
    ma_window=ma_window,
    rebuild=rebuild_signal,
)

print(f"Signal shape: {signal.shape[0]} dates x {signal.shape[1]} stocks")
print(f"Signal true cells: {int(signal.to_numpy().sum())}")

del daily
gc.collect()

Signal shape: 1094 dates x 2052 stocks
Signal true cells: 949415


0

### get minute kbar data

In [ ]:
signal = pd.read_parquet(signal_output)
signal.index = pd.to_datetime(signal.index)

no_price_pairs = load_no_price_pairs(daily_output)

# 3. Build supplemental K bars. Full kbar/1min is primary; above_ma60 stores only missing symbols needed for this study.
signal_desc = signal.sort_index(ascending=False)

bar = tqdm(signal_desc.iterrows(), total=len(signal_desc), desc="Build filtered kbar")
for trade_date, signal_row in bar:
    stock_ids = signal_row_to_stock_ids(signal_row)
    date_str = pd.Timestamp(trade_date).strftime("%Y-%m-%d")
    bar.set_postfix_str(f"{date_str} check")

    if not stock_ids:
        bar.set_postfix_str(f"{date_str} empty")
        continue

    full_file = full_kbar_dir / f"{date_str}.parquet"
    supplement_file = kbar_output_dir / f"{date_str}.parquet"

    if full_file.exists():
        full_ids = read_kbar_ids(full_file)
    else:
        # Full-day kbar is unavailable; still fetch the strategy-required symbols
        # into above_ma60 so the study can run without a complete full kbar file.
        update_missing_full_kbar_dates(missing_full_kbar_output, date_str)
        full_ids = set()
        bar.set_postfix_str(f"{date_str} full missing; supplement")

    known_missing_ids = get_missing_kbar_ids(missing_kbar_output, date_str)

    expected_ids = set(stock_ids) - known_missing_ids
    if not expected_ids:
        bar.set_postfix_str(f"{date_str} all missing-known")
        continue

    supplement_ids = read_kbar_ids(supplement_file)
    available_ids = full_ids | supplement_ids

    api_ids = expected_ids - available_ids
    if not api_ids:
        bar.set_postfix_str(f"{date_str} covered")
        continue

    no_price_ids = {
        stock_id for stock_id in api_ids
        if (date_str, stock_id) in no_price_pairs
    }
    if no_price_ids:
        update_missing_kbar_ids(missing_kbar_output, date_str, no_price_ids)
        api_ids -= no_price_ids
        bar.set_postfix_str(f"{date_str} no price {len(no_price_ids)}")
        if not api_ids:
            continue

    api_batches = chunk_list(sorted(api_ids), api_batch_size)
    for batch_idx, api_batch in enumerate(api_batches, start=1):
        bar.set_postfix_str(
            f"{date_str} api {batch_idx}/{len(api_batches)} ({len(api_batch)})"
        )
        try:
            report = client.get_multi_kbar(
                start_date=date_str,
                end_date=date_str,
                output_dir=kbar_output_dir,
                stock_ids=api_batch,
                max_retries=1,
                retry_wait=retry_wait,
                rate_limit_wait=rate_limit_wait,
                rate_limit_max_retries=rate_limit_max_retries,
                return_report=True,
                verbose=False,
                show_progress=False,
                status_callback=lambda msg: bar.set_postfix_str(msg),
            )
            if report is False or report.get("rate_limited"):
                bar.set_postfix_str(f"stopped {date_str}")
                print(f"Stopped at {date_str}; rerun this cell later to resume.")
                break

            missing_from_download = report.get("missing_stock_ids_by_date", {}).get(date_str, [])
            if missing_from_download:
                update_missing_kbar_ids(missing_kbar_output, date_str, missing_from_download)
                bar.set_postfix_str(f"{date_str} missing {len(missing_from_download)}")
            else:
                bar.set_postfix_str(f"{date_str} api batch done")
        except Exception as exc:
            bar.set_postfix_str(f"{date_str} failed: {type(exc).__name__}")
            print(f"{date_str} failed: {type(exc).__name__}: {exc}")
            time.sleep(retry_wait)
    else:
        continue
    break

Build filtered kbar:  90%|████████▉ | 980/1094 [13:38:20<1:35:11, 50.10s/it, 2022-06-27 rate limit 1/2 wait 3660s]  


KeyboardInterrupt: 

: 

In [ ]:
report = client.get_multi_kbar(
            start_date=start_date,
            end_date=None,
            output_dir=kbar_output_dir,
            stock_ids=None,
            max_retries=1,
            retry_wait=retry_wait,
            rate_limit_wait=rate_limit_wait,
            rate_limit_max_retries=rate_limit_max_retries,
            return_report=True,
            verbose=True,
            show_progress=True,
        )

取得全部台股清單...

⚠ taiwan_stock_info rate limit (1/5)，等待 3660s 後重試。錯誤: 'data'
股票數: 2674
取得交易日列表...
交易日數: 1096
已完成: 831 天 / 待下載: 265 天
預估剩餘: 11.0 小時


Build filtered kbar:   0%|          | 0/265 [12:44:41<?, ?day/s, ✗ 2026-07-10 失敗]


KeyboardInterrupt: 

: 

In [10]:
# Optional inspection
# signal = pd.read_parquet(TW_STOCK_KBAR_1MIN / "2025-03-05.parquet")
signal = pd.read_parquet('/Users/xinc/GitHub/google_drive/Data/trading/tw_stock/kbar/1H/2023-03-13.parquet')
signal

,date,hour,stock_id,open,high,low,close,volume
0,2023-03-13,09:00:00,2317,102.0,102.0,101.5,102.0,4443
1,2023-03-13,09:00:00,2330,513.0,515.0,509.0,510.0,6379
2,2023-03-13,10:00:00,2317,102.0,103.5,101.5,103.0,9935
3,2023-03-13,10:00:00,2330,510.0,516.0,510.0,515.0,3323
4,2023-03-13,11:00:00,2317,103.0,103.0,102.5,103.0,1003
5,2023-03-13,11:00:00,2330,515.0,516.0,514.0,515.0,2154
6,2023-03-13,12:00:00,2317,103.0,103.0,102.5,102.5,1984
7,2023-03-13,12:00:00,2330,516.0,517.0,515.0,517.0,2802
8,2023-03-13,13:00:00,2317,103.0,103.0,102.5,103.0,6154
9,2023-03-13,13:00:00,2330,516.0,518.0,516.0,516.0,6758


### inspection

In [ ]:
# Check whether the MA60 signal universe is covered by full kbar + supplement files.
signal = pd.read_parquet(signal_output)
signal.index = pd.to_datetime(signal.index)

coverage = build_kbar_coverage_report(
    signal=signal,
    full_kbar_dir=full_kbar_dir,
    supplement_dir=kbar_output_dir,
)
missing_full_top, incomplete_full_top = summarize_kbar_coverage(coverage, top_n=30)

missing_full_count = int((~coverage["full_exists"]).sum())
incomplete_full_count = int((coverage["full_exists"] & (coverage["missing_count"] > 0)).sum())

print(f"Coverage dates: {len(coverage):,}")
print(f"Missing full kbar dates: {missing_full_count:,}")
print(f"Full exists but signal still missing kbar: {incomplete_full_count:,}")

print("\nTop missing full kbar dates:")
display(missing_full_top)

print("\nTop incomplete full kbar dates:")
display(incomplete_full_top)

# Inspect one date by setting inspect_date, for example: inspect_date = "2025-02-14"
inspect_date = None
if inspect_date is not None:
    inspect_row = coverage.loc[coverage["date"] == inspect_date].iloc[0]
    inspect_missing_ids = inspect_row["missing_ids"]
    print(f"{inspect_date} missing ids: {len(inspect_missing_ids):,}")
    print(inspect_missing_ids[:100])

Coverage dates: 1,094
Missing full kbar dates: 942
Full exists but signal still missing kbar: 120

Top missing full kbar dates:


,date,signal_count,full_exists,full_count,supplement_count,covered_count,missing_count
1093,2026-07-13,941,False,0,941,941,0
1092,2026-07-09,983,False,0,972,972,11
1091,2026-07-08,1015,False,0,1004,1004,11
1090,2026-07-07,1204,False,0,1204,1204,0
1089,2026-07-06,1193,False,0,1193,1193,0
1088,2026-07-03,1052,False,0,1052,1052,0
1087,2026-07-02,968,False,0,968,968,0
1086,2026-07-01,978,False,0,978,978,0
1085,2026-06-30,898,False,0,898,898,0
1084,2026-06-29,829,False,0,829,829,0



Top incomplete full kbar dates:


,date,signal_count,full_exists,full_count,supplement_count,covered_count,missing_count
764,2025-03-06,1246,True,2179,0,1194,52
763,2025-03-05,1165,True,2177,0,1116,49
594,2024-06-19,1267,True,2128,0,1219,48
836,2025-06-20,762,True,2210,0,715,47
751,2025-02-14,1040,True,106,937,993,47
885,2025-08-28,1017,True,2229,0,970,47
691,2024-11-12,762,True,2144,0,715,47
860,2025-07-24,752,True,2216,0,706,46
861,2025-07-25,752,True,2214,0,706,46
884,2025-08-27,971,True,2240,0,925,46


## backtest

### resample

In [ ]:
from resample_kbar import resample_kbar_to_1h
from cloud_data import TW_STOCK_KBAR_1MIN, TW_STOCK_KBAR_ABOVE_MA60, TW_STOCK_KBAR_1H

resample_kbar_to_1h(
    source_dir=TW_STOCK_KBAR_1MIN,
    output_dir=TW_STOCK_KBAR_1H,
    supplement_dir=TW_STOCK_KBAR_ABOVE_MA60,
    start_date="2022-01-01",
    end_date=None
)

待處理: 1094 天（已有 0 天）


Resample 1min→1H: 100%|██████████| 1094/1094 [40:51<00:00,  2.24s/day]

完成: 1094 天，錯誤: 0 天


['2022-01-03',
 '2022-01-04',
 '2022-01-05',
 '2022-01-06',
 '2022-01-07',
 '2022-01-10',
 '2022-01-11',
 '2022-01-12',
 '2022-01-13',
 '2022-01-14',
 '2022-01-17',
 '2022-01-18',
 '2022-01-19',
 '2022-01-20',
 '2022-01-21',
 '2022-01-24',
 '2022-01-25',
 '2022-01-26',
 '2022-02-07',
 '2022-02-08',
 '2022-02-09',
 '2022-02-10',
 '2022-02-11',
 '2022-02-14',
 '2022-02-15',
 '2022-02-16',
 '2022-02-17',
 '2022-02-18',
 '2022-02-21',
 '2022-02-22',
 '2022-02-23',
 '2022-02-24',
 '2022-02-25',
 '2022-03-01',
 '2022-03-02',
 '2022-03-03',
 '2022-03-04',
 '2022-03-07',
 '2022-03-08',
 '2022-03-09',
 '2022-03-10',
 '2022-03-11',
 '2022-03-14',
 '2022-03-15',
 '2022-03-16',
 '2022-03-17',
 '2022-03-18',
 '2022-03-21',
 '2022-03-22',
 '2022-03-23',
 '2022-03-24',
 '2022-03-25',
 '2022-03-28',
 '2022-03-29',
 '2022-03-30',
 '2022-03-31',
 '2022-04-01',
 '2022-04-06',
 '2022-04-07',
 '2022-04-08',
 '2022-04-11',
 '2022-04-12',
 '2022-04-13',
 '2022-04-14',
 '2022-04-15',
 '2022-04-18',
 '2022-04-

#### inspection

In [ ]:
# 檢查是不是所有股票都被 resample

from resample_kbar import _read_and_merge

output_files = sorted(TW_STOCK_KBAR_1H.glob("*.parquet"))
rows = []
for f in tqdm(output_files, desc="checking"):
    d = f.stem
    src = _read_and_merge(d, TW_STOCK_KBAR_1MIN, TW_STOCK_KBAR_ABOVE_MA60)
    src_n = src["stock_id"].nunique() if not src.empty else 0
    out_n = pd.read_parquet(f, columns=["stock_id"])["stock_id"].nunique()
    rows.append({"date": d, "src_stocks": src_n, "out_stocks": out_n, "diff": src_n - out_n})

report = pd.DataFrame(rows)
mismatch = report[report["diff"] != 0]
print(f"總天數: {len(report)}, 不符天數: {len(mismatch)}")
display(mismatch.sort_values("diff", ascending=False).head(20))


checking: 100%|██████████| 1094/1094 [00:22<00:00, 49.52it/s]

總天數: 1094, 不符天數: 0


,date,src_stocks,out_stocks,diff


### get signal df

In [ ]:
from cloud_data import TW_STOCK_KBAR_1H
from backtest import BacktestConfig, build_signal_df_from_directory

config = BacktestConfig(
    max_ma55_distance=0.25,
    support_tolerance=0.03,
    vol_ma_window=20,
    pullback_volume_ratio=0.70,
    platform_lookback=12,
    platform_max_range=0.08,
)

feature_df = build_signal_df_from_directory(
    kbar_dir=TW_STOCK_KBAR_1H,
    start_date="2022-01-01",
    end_date=None,
    config=config,
    output_file="signal_df.parquet",
    include_all_rows=True
)

feature_df

,date,hour,stock_id,open,high,low,close,volume,datetime,ma5,...,volume_contract,stop_falling,platform_range_ok,platform_flat,platform_tight,trend_filter,first_entry_signal,confirm_signal,entry_signal,raw_stop_signal
0,2026-04-28,09:00:00,00400A,12.75,12.88,12.73,12.73,28722,2026-04-28 09:00:00,NaN,...,False,False,False,False,False,False,False,False,False,False
1,2026-04-28,10:00:00,00400A,12.74,12.81,12.69,12.79,11728,2026-04-28 10:00:00,NaN,...,False,True,False,False,False,False,False,False,False,False
2,2026-04-28,11:00:00,00400A,12.79,12.80,12.72,12.74,6076,2026-04-28 11:00:00,NaN,...,False,False,False,False,False,False,False,False,False,False
3,2026-04-28,12:00:00,00400A,12.74,12.82,12.73,12.79,8010,2026-04-28 12:00:00,NaN,...,False,True,False,False,False,False,False,False,False,False
4,2026-04-28,13:00:00,00400A,12.79,12.80,12.74,12.78,4226,2026-04-28 13:00:00,12.766,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5012886,2026-07-13,09:00:00,9962,9.80,9.80,9.80,9.80,1,2026-07-13 09:00:00,9.858,...,True,True,True,True,True,False,False,False,False,False
5012887,2026-07-13,10:00:00,9962,9.80,9.80,9.79,9.80,24,2026-07-13 10:00:00,9.842,...,False,True,True,True,True,False,False,False,False,False
5012888,2026-07-13,11:00:00,9962,9.88,9.89,9.88,9.89,8,2026-07-13 11:00:00,9.844,...,True,True,True,True,True,False,False,False,False,False
5012889,2026-07-13,12:00:00,9962,9.85,9.85,9.79,9.85,13,2026-07-13 12:00:00,9.844,...,False,True,True,True,True,False,False,False,False,False


In [ ]:
from backtest import run_signal_backtest

trades_df, equity_df = run_signal_backtest(
    feature_df,
    config=config,
)

equity_df[["datetime", "active_positions", "portfolio_return", "nav"]].tail()

,datetime,active_positions,portfolio_return,nav
4776,2026-07-13 09:00:00,153,-0.008890,0.670006
4777,2026-07-13 10:00:00,144,0.000787,0.670533
4778,2026-07-13 11:00:00,119,-0.003866,0.667941
4779,2026-07-13 12:00:00,124,-0.001352,0.667037
4780,2026-07-13 13:00:00,116,0.002066,0.668415


In [14]:
trades_df

,date,hour,stock_id,open,high,low,close,volume,datetime,ma5,...,confirm_signal,entry_signal,raw_stop_signal,entry_order,exit_order,position,active_bar,strategy_return,entered,exited
0,2022-01-03,09:00:00,0050,146.00,147.35,146.00,146.40,3248,2022-01-03 09:00:00,NaN,...,False,False,False,False,False,False,False,0.00000,False,False
1,2022-01-03,09:00:00,0051,60.90,61.30,60.85,60.85,25,2022-01-03 09:00:00,NaN,...,False,False,False,False,False,False,False,0.00000,False,False
2,2022-01-03,09:00:00,0052,134.65,135.95,134.65,135.20,373,2022-01-03 09:00:00,NaN,...,False,False,False,False,False,False,False,0.00000,False,False
3,2022-01-03,09:00:00,0053,70.25,70.75,70.25,70.25,24,2022-01-03 09:00:00,NaN,...,False,False,False,False,False,False,False,0.00000,False,False
4,2022-01-03,09:00:00,0054,31.68,31.78,31.68,31.78,2,2022-01-03 09:00:00,NaN,...,False,False,False,False,False,False,False,0.00000,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5012886,2026-07-13,13:00:00,9946,19.05,19.05,18.70,18.70,160,2026-07-13 13:00:00,18.970,...,False,False,True,False,True,False,False,0.00000,False,False
5012887,2026-07-13,13:00:00,9949,26.65,26.70,26.55,26.55,58,2026-07-13 13:00:00,28.070,...,False,False,False,False,False,False,False,0.00000,False,False
5012888,2026-07-13,13:00:00,9958,117.50,117.50,116.50,116.50,346,2026-07-13 13:00:00,117.100,...,False,False,True,False,True,False,True,0.00086,False,True
5012889,2026-07-13,13:00:00,9960,32.80,32.80,32.15,32.15,7,2026-07-13 13:00:00,32.750,...,False,False,True,False,True,False,False,0.00000,False,False
